# Set up Colab environment

Adapted from Programming Assignment 3

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import RandAugment
from IPython.core.debugger import set_trace

import transformers
# added specific functions for ease of calling
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from datasets import load_dataset

from einops import rearrange
from einops.layers.torch import Rearrange
from datasets import load_dataset

import os
import sys
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

# Used for saving model results
from datetime import datetime
from pathlib import Path
import csv
import random

import umap

from google.colab import drive

torch.manual_seed(0)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(F"Device set to {device}")

In [ ]:
drive.mount('/content/gdrive')

# Add a shortcut to the project folder in your Drive home
base_dir = "/content/gdrive/MyDrive/CS4782 Project"
sys.path.append(base_dir)

# Define result dir
Uses the current date/time by default, but change this to a pre-existing folder name if you want to resume from a saved checkpoint

In [ ]:
result_dir = f"{base_dir}/results/{datetime.now()}"
# result_dir = f"{base_dir}/results/[insert directory name here]"
Path(result_dir).mkdir(parents=True, exist_ok=True)

# Define hyperparameters

In [ ]:
# Add more as needed

epochs = 10              # was 60 in the paper, doing 10 to save compute
max_padding = 512        # was 16 for testing
n_classes = 2

# Print intermediate results every x batches
print_batches = 200

# DataLoader
batch_size = 16         # was 32 for testing
shuffle_data = True

# AdamW Optimizer + LR schedule (LoRA paper for GLUE: 6% warmup, linear decay)
learning_rate      = 5e-4
weight_decay       = 0.01
warmup_ratio       = 0.06
grad_clip          = 1.0
classifier_dropout = 0.1

# LoRA
rank       = 6
lora_alpha = 8         # was 16 for testing

# Model classes

In [ ]:
class LoRA_Adapter(nn.Module):
  """
  Wraps a frozen nn.Linear and adds a trainable low-rank update.
  Forward pass: h = W0*x + scaling * B(A*x)
  A is Gaussian-initialized, B is zero-initialized so the update
  starts at zero at the beginning of training.
  """
  def __init__(self, linear: nn.Linear, rank: int, lora_alpha: int = None):
    """
    linear     : pretrained nn.Linear layer to wrap and freeze
    rank       : rank r of the low-rank decomposition
    lora_alpha : scaling constant alpha, defaults to rank if not set
    """
    super().__init__()

    self.rank       = rank
    self.lora_alpha = lora_alpha if lora_alpha is not None else rank
    self.scaling    = self.lora_alpha / self.rank

    in_features  = linear.in_features
    out_features = linear.out_features

    # Freeze all parameters in the pretrained layer
    self.pretrained_linear = linear
    # for param in self.pretrained_linear.parameters():
    #     param.requires_grad = False

    # A shape: (rank, in_features): initialized with Gaussian noise
    # B shape: (out_features, rank): initialized to zeros
    self.lora_A = nn.Parameter(torch.empty(rank, in_features))
    self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

    nn.init.normal_(self.lora_A, mean=0.0, std=1 / math.sqrt(rank))

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    """
    Computes h = W0*x + scaling * B(A*x).
    x shape: (..., in_features)
    output shape: (..., out_features)
    """
    # Frozen pretrained path
    pretrained_out = self.pretrained_linear(x)

    # Low-rank trainable path, scaled by alpha/r
    lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B)
    lora_out = lora_out * self.scaling

    return pretrained_out + lora_out

  def count_trainable_params(self) -> int:
    """Returns the number of trainable parameters in this layer (A and B only)."""
    return sum(p.numel() for p in [self.lora_A, self.lora_B])


In [ ]:
class LoRA_RoBERTa_Adapter(nn.Module):
  def __init__(self, pretrained: transformers.models.roberta.modeling_roberta.RobertaModel, rank: int, lora_alpha: int = None):
    """
    pretrained : pretrained RoBERTa model to apply LoRA on
    rank       : rank r of the low-rank decomposition
    lora_alpha : scaling constant alpha, defaults to rank if not set
    """
    super().__init__()

    self.pretrained = pretrained

    # Wrap Q, K, V on every layer first, doing this before the freeze step is important, if we freeze first and
    # then re-run this cell, the freeze loop will sweep up the existing LoRA params and freeze them too.
    for layer in pretrained.encoder.layer:
      # Failsafe for rerunning code during testing, avoid wrapping weight matrices in LoRA adapter twice
      if (type(layer.attention.self.query) == LoRA_Adapter):
        continue
      layer.attention.self.query = LoRA_Adapter(layer.attention.self.query, rank, lora_alpha)
      layer.attention.self.key = LoRA_Adapter(layer.attention.self.key, rank, lora_alpha)
      layer.attention.self.value = LoRA_Adapter(layer.attention.self.value, rank, lora_alpha)

    # Now freeze everything except the LoRA params. Pooler stays frozen because the new classifier doesn't use it.
    for name, param in pretrained.named_parameters():
      param.requires_grad = ('lora_A' in name) or ('lora_B' in name)

  def forward(self, x: torch.Tensor, attention_mask) -> torch.Tensor:
    """
    x shape: (..., sequence_length)
    output shape: (..., sequence_length)
    """
    return self.pretrained(x, attention_mask=attention_mask)


In [ ]:
class RoBERTa_Classifier(nn.Module):
  """
  Standard RoBERTa classification head (matches HF RobertaClassificationHead):
    <s> token -> dropout -> dense(768->768) -> tanh -> dropout -> out_proj(768->n)
  We use last_hidden_state[:, 0] directly and skip the (randomly-initialized) pooler.
  """
  def __init__(self, roberta: nn.Module, n_classes: int, dropout_p: float = 0.1):
    super().__init__()

    self.roberta  = roberta
    self.dropout1 = nn.Dropout(dropout_p)
    self.dense    = nn.Linear(768, 768)
    self.dropout2 = nn.Dropout(dropout_p)
    self.out_proj = nn.Linear(768, n_classes)

  def forward(self, x: torch.Tensor, attention_mask) -> torch.Tensor:
    out = self.roberta(x, attention_mask=attention_mask)
    h   = out.last_hidden_state[:, 0]   # token representation
    h   = self.dropout1(h)
    h   = torch.tanh(self.dense(h))
    h   = self.dropout2(h)
    return self.out_proj(h)


# Download pre-trained RoBERTa model, SST-2 dataset

In [ ]:
# Putting all IDs here so we can easily switch them out for testing
model_id = "FacebookAI/roberta-base"
dataset_id = "stanfordnlp/sst2"

In [ ]:
sst2_dataset = load_dataset(dataset_id)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
roberta_pretrained = AutoModel.from_pretrained(model_id)

# Define training/evaluation cycle

In [ ]:
def training_cycle(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                   tokenizer: transformers.models.roberta.tokenization_roberta.RobertaTokenizer,
                   optimizer: torch.optim.Optimizer, scheduler, criterion: torch.nn,
                   epochs: int, model_name: str, grad_clip: float = 1.0):
  """
  model:      nn.Module model to train
  train_loader, val_loader: data loaders
  tokenizer:  tokenizer
  optimizer:  AdamW
  scheduler:  LR scheduler (stepped per batch)
  criterion:  loss
  epochs:     number of epochs
  grad_clip:  max grad norm

  Returns three lists: training loss, validation loss, and validation accuracy per epoch.
  """

  checkpoint_path = f"{result_dir}/{model_name}_checkpoint.pth"

  start_epoch = 1
  best_val_loss = float('inf')

  training_loss_per_epoch = []
  val_loss_per_epoch = []
  val_acc_per_epoch = []

  # Restore checkpoint if it exists
  if os.path.exists(checkpoint_path):
    print("Restoring training state from checkpoint")

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]

    training_loss_per_epoch = checkpoint["training_loss_per_epoch"]
    val_loss_per_epoch = checkpoint["val_loss_per_epoch"]
    val_acc_per_epoch = checkpoint["val_acc_per_epoch"]

    torch.set_rng_state(checkpoint["torch_rng_state"].cpu().byte())
    if torch.cuda.is_available():
        rng_states = [rng.cpu() for rng in checkpoint["cuda_rng_state"]]
        torch.cuda.set_rng_state_all(rng_states)

    np.random.set_state(checkpoint["numpy_rng_state"])
    random.setstate(checkpoint["python_rng_state"])

    print(f"Resuming from epoch {start_epoch}")

  trainable = [p for p in model.parameters() if p.requires_grad]

  for epoch in range(start_epoch, epochs + 1):
    total_training_loss = 0.
    total_val_loss = 0.
    val_acc = 0.

    # Training
    model.train()
    for (batch_idx, batch) in enumerate(train_loader):
      optimizer.zero_grad()

      sentence = batch['sentence']
      tokenized = tokenizer(sentence, padding='max_length', max_length=max_padding,
                            truncation=True, return_tensors='pt').to(device)
      label = batch['label'].to(device)

      pred = model(tokenized['input_ids'], tokenized['attention_mask'])

      loss = criterion(pred, label)
      total_training_loss += loss.item() * label.size(0)      # Multiply by size of current batch -- last batch may be smaller than batch_size

      loss.backward()
      torch.nn.utils.clip_grad_norm_(trainable, max_norm=grad_clip)
      optimizer.step()
      scheduler.step()

      if batch_idx % print_batches == 0:
        cur_lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch} | Batch {batch_idx+1}/{len(train_loader)}: Training loss = {loss.item():.4f}  lr = {cur_lr:.2e}")

    total_training_loss /= len(train_loader.dataset)          # Divide by number of samples to get average loss
    training_loss_per_epoch.append(total_training_loss)
    print(f"Epoch {epoch}: Training loss = {total_training_loss:.4f}")

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
      for (batch_idx, batch) in enumerate(val_loader):
        sentence = batch['sentence']
        tokenized = tokenizer(sentence, padding='max_length', max_length=max_padding,
                              truncation=True, return_tensors='pt').to(device)
        label = batch['label'].to(device)

        pred = model(tokenized['input_ids'], tokenized['attention_mask'])

        loss = criterion(pred, label)
        total_val_loss += loss.item() * label.size(0)
        correct += (pred.argmax(dim=-1) == label).sum().item()
        total   += label.size(0)

        if batch_idx % print_batches == 0:
          print(f"Epoch {epoch} | Batch {batch_idx+1}/{len(val_loader)}: Validation loss = {loss.item():.4f}")

      total_val_loss /= len(val_loader.dataset)
      val_acc = correct / total

      val_loss_per_epoch.append(total_val_loss)
      val_acc_per_epoch.append(val_acc)

      print(f"Epoch {epoch}: Validation loss = {total_val_loss:.4f}  Val acc = {val_acc:.4f}")

      # Save best model
      if total_val_loss < best_val_loss:
        best_val_loss = total_val_loss
        torch.save(model.state_dict(), f'{result_dir}/{model_name}_best_model.pth')
        print(f"New best model saved at epoch {epoch}")

    # Save checkpoint
    checkpoint = {
      "epoch": epoch,
      "model_state_dict": model.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
      "scheduler_state_dict": scheduler.state_dict(),
      "best_val_loss": best_val_loss,
      "training_loss_per_epoch": training_loss_per_epoch,
      "val_loss_per_epoch": val_loss_per_epoch,
      "val_acc_per_epoch": val_acc_per_epoch,
      "torch_rng_state": torch.get_rng_state(),
      "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
      "numpy_rng_state": np.random.get_state(),
      "python_rng_state": random.getstate(),
    }

    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved after epoch {epoch}")

  return training_loss_per_epoch, val_loss_per_epoch, val_acc_per_epoch


In [ ]:
def evaluate(model: nn.Module, test_loader: DataLoader,
                   tokenizer: transformers.models.roberta.tokenization_roberta.RobertaTokenizer,
                   criterion: torch.nn):
  """
  model:        nn.Module model to evaluate
  test_loader:  testing data loader
  tokenizer:    tokenizer
  criterion:    loss criterion

  Returns the testing loss.
  """

  model.eval()
  total_loss = 0.
  correct, total = 0, 0

  with torch.no_grad():
    for (batch_idx, batch) in enumerate(test_loader):
      sentence = batch['sentence']
      tokenized = tokenizer(sentence, padding='max_length', max_length=max_padding,
                            truncation=True, return_tensors='pt').to(device)
      label = batch['label'].to(device)

      pred = model(tokenized['input_ids'], tokenized['attention_mask'])

      loss = criterion(pred, label)
      total_loss += loss.item() * label.size(0)

      pred_class = pred.argmax(dim=-1)
      correct += (pred_class == label).sum().item()
      total += label.size(0)

      if batch_idx % print_batches == 0:
          print(f"Batch {batch_idx}: Testing loss = {loss.item():.4f}")

  total_loss /= len(test_loader.dataset)
  test_acc = correct/total

  return total_loss, test_acc


# Train LoRA model

In [ ]:
# After looking at the HF SST2 dataset, test split has hidden labels (all -1) which is useless for accuracy.
# Carve a held out val from train, and use the original validation split as our test set.
from torch.utils.data import random_split

full_train = sst2_dataset['train']
val_carve  = 2000
gen        = torch.Generator().manual_seed(0)
train_split, val_split = random_split(
    full_train, [len(full_train) - val_carve, val_carve], generator=gen
)

train_dataloader = DataLoader(
    train_split,
    batch_size=batch_size,
    shuffle=shuffle_data
)

val_dataloader = DataLoader(
    val_split,
    batch_size=batch_size,
    shuffle=False
)

test_dataloader = DataLoader(
    sst2_dataset['validation'],   # the real labeled held out set
    batch_size=batch_size,
    shuffle=False
)

print(f"train: {len(train_split)}  val: {len(val_split)}  test: {len(sst2_dataset['validation'])}")


In [ ]:
lora_model = LoRA_RoBERTa_Adapter(roberta_pretrained, rank, lora_alpha)
lora_classifier = RoBERTa_Classifier(lora_model, n_classes, dropout_p=classifier_dropout).to(device)

# Should see around 76 trainable tensors (12 layers * 3 mats * {A, B} = 72 LoRA + 4 head)
trainable_names = [n for n, p in lora_classifier.named_parameters() if p.requires_grad]
trainable_count = sum(p.numel() for p in lora_classifier.parameters() if p.requires_grad)
total_count     = sum(p.numel() for p in lora_classifier.parameters())
print(f"Trainable tensors: {len(trainable_names)} (expect 72 LoRA + 4 head = 76)")
print(f"Trainable params:  {trainable_count:,} / {total_count:,}  ({100*trainable_count/total_count:.3f}%)")

# Optimizer: only the params that actually need updating
trainable_params = [p for p in lora_classifier.parameters() if p.requires_grad]
lora_optimizer = optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)

# Linear warmup + linear decay (matches LoRA paper for GLUE)
total_steps  = len(train_dataloader) * epochs
warmup_steps = int(warmup_ratio * total_steps)
lr_scheduler = get_linear_schedule_with_warmup(
    lora_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
print(f"LR schedule: {warmup_steps} warmup steps / {total_steps} total")

# Cross entropy loss function
criterion = torch.nn.CrossEntropyLoss()


In [ ]:
train_loss, val_loss, val_acc = training_cycle(
    lora_classifier, train_dataloader, val_dataloader,
    tokenizer, lora_optimizer, lr_scheduler, criterion,
    epochs, "LoRA", grad_clip=grad_clip
)


In [ ]:
with open(f'{result_dir}/LoRA_training.csv', 'w', newline='') as f:
  wr = csv.writer(f, quoting=csv.QUOTE_MINIMAL)

  # Header row
  wr.writerow(['Epoch', 'Training loss', 'Validation loss', 'Validation accuracy'])

  # Data rows
  for i, (tr, vl, va) in enumerate(zip(train_loss, val_loss, val_acc), start=1):
      wr.writerow([i, tr, vl, va])

# Evaluate LoRA model

In [ ]:
# load best model so we dont need to retrain
lora_classifier.load_state_dict(torch.load(f'{result_dir}/LoRA_best_model.pth'))

In [ ]:
test_loss, test_acc = evaluate(lora_classifier, test_dataloader, tokenizer, criterion)

print(f"Testing loss: {test_loss:.4f}")
print(f"Testing accuracy: {test_acc:.4f}")

In [ ]:
with open(f'{result_dir}/LoRA_testing.csv', 'w', newline='') as f:
  wr = csv.writer(f, quoting=csv.QUOTE_MINIMAL)

  wr.writerow(['Testing loss', 'Testing accuracy'])
  wr.writerow([test_loss, test_acc])